In [1]:
import torch


def apply_masks(x, masks, concat=True):
    """
    :param x: tensor of shape [B (batch-size), N (num-patches), D (feature-dim)]
    :param masks: list of tensors of shape [B, K] containing indices of K patches in [N] to keep
    """
    all_x = []
    for m in masks:
        mask_keep = m.unsqueeze(-1).repeat(1, 1, x.size(-1))
        all_x += [torch.gather(x, dim=1, index=mask_keep)]
    if not concat:
        return all_x

    return torch.cat(all_x, dim=0)

In [2]:
def get_complement_masks(masks, n):
    """
    masks = [b, k] contains k patch indices in [0, n)
    Returns complement = [b, n-k] containing the remaining indices
    """
    b, k = masks.shape
    device = masks.device

    # Start with all indices marked as "keep" (True)
    keep = torch.ones(b, n, dtype=torch.bool, device=device)
    # Mark the masked indices as False (these are the ones we want to remove)
    keep.scatter_(1, masks, False)

    # The complement indices are those still marked True
    complement = torch.arange(n, device=device).unsqueeze(0).expand(b, -1)
    complement_indices = complement[keep].view(b, n - k)

    return complement_indices

In [3]:
import torch

x = torch.arange(1, 28).reshape(3, 3, 3)

print(x)

tensor([[[ 1,  2,  3],
         [ 4,  5,  6],
         [ 7,  8,  9]],

        [[10, 11, 12],
         [13, 14, 15],
         [16, 17, 18]],

        [[19, 20, 21],
         [22, 23, 24],
         [25, 26, 27]]])


In [4]:
y = torch.tensor([
    [0,1], [0,2], [1,0]
])
y.shape

torch.Size([3, 2])

In [5]:
y = [y]

In [6]:
y

[tensor([[0, 1],
         [0, 2],
         [1, 0]])]

In [7]:
z = torch.tensor([
    [2],[1],[2]
])

In [8]:
masked = apply_masks(x, y)

In [9]:
masked

tensor([[[ 1,  2,  3],
         [ 4,  5,  6]],

        [[10, 11, 12],
         [16, 17, 18]],

        [[22, 23, 24],
         [19, 20, 21]]])

In [10]:
complement = get_complement_masks(y[0],3)
complement

tensor([[2],
        [1],
        [2]])

In [11]:
masked_complement = apply_masks(x, [complement])
masked_complement

tensor([[[ 7,  8,  9]],

        [[13, 14, 15]],

        [[25, 26, 27]]])

In [12]:
y = torch.cat(y, dim=0)
y

tensor([[0, 1],
        [0, 2],
        [1, 0]])

In [13]:
y.shape

torch.Size([3, 2])

In [20]:
from utils.patch_embed import PatchEmbed
def load_image_as_tensor(image_path):
    """Loads a real image, converts to RGB, and normalizes to [0, 1]."""
    image = Image.open(image_path).convert("RGB")
    # Convert to numpy array then to tensor
    np_img = np.array(image, dtype=np.float32) / 255.0
    # Permute to [C, H, W]
    tensor = torch.tensor(np_img).permute(2, 0, 1)
    return tensor

image_tensor = load_image_as_tensor("image.png")
image_tensor = image_tensor.unsqueeze(0)
print(image_tensor.shape)  # Should be [1, 3, H, W]
C, H, W = image_tensor.shape[1:]
print(f"Original Image Shape: {image_tensor.shape} (B, C, H, W)")
PATCH_SIZE = 16
EMBED_DIM = 768
n_patches_h = H // PATCH_SIZE
n_patches_w = W // PATCH_SIZE
total_patches = n_patches_h * n_patches_w

print(f"Patch Grid: {n_patches_h} x {n_patches_w} = {total_patches} total patches")

patch_embed = PatchEmbed()
x = patch_embed(image_tensor)

torch.Size([1, 3, 663, 1455])
Original Image Shape: torch.Size([1, 3, 663, 1455]) (B, C, H, W)
Patch Grid: 41 x 90 = 3690 total patches


In [21]:
x.shape

torch.Size([1, 3690, 768])

In [22]:
masked_indices = torch.randperm(total_patches)[:2690]
masked_indices = masked_indices.unsqueeze(0)
masked_indices.shape

torch.Size([1, 2690])

In [23]:
from model import Model
model = Model(num_patches=total_patches, embed_dim=EMBED_DIM)
out, loss = model(x, masks=masked_indices, train=True)

In [24]:
out.shape

torch.Size([1, 1000, 768])

In [25]:
loss

tensor(0.3343, grad_fn=<MseLossBackward0>)

In [27]:
pip install torchinfo

  Using cached torchinfo-1.8.0-py3-none-any.whl.metadata (21 kB)
Using cached torchinfo-1.8.0-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.


In [28]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                   Param #
Model                                    2,833,920
├─VisionTransformer: 1-1                 --
│    └─ModuleList: 2-1                   --
│    │    └─Block: 3-1                   7,087,872
│    │    └─Block: 3-2                   7,087,872
│    │    └─Block: 3-3                   7,087,872
│    │    └─Block: 3-4                   7,087,872
│    │    └─Block: 3-5                   7,087,872
│    │    └─Block: 3-6                   7,087,872
│    │    └─Block: 3-7                   7,087,872
│    │    └─Block: 3-8                   7,087,872
│    │    └─Block: 3-9                   7,087,872
│    │    └─Block: 3-10                  7,087,872
│    │    └─Block: 3-11                  7,087,872
│    │    └─Block: 3-12                  7,087,872
│    └─LayerNorm: 2-2                    1,536
├─VisionTransformerPredictor: 1-2        --
│    └─Linear: 2-3                       295,296
│    └─ModuleList: 2-4                   --
│    │    └─Bloc